### learned sparsity tables ###

In [168]:
# Python 3 code für Jupyter Notebook
import os
import glob
import re
from collections import defaultdict, OrderedDict
import pandas as pd

# ---------- Hilfsfunktionen ----------
def parse_result_lines(path):
    """
    Erwartetes Format:
    run; test_dataset; aAcc_mean; aAcc_std; mIoU_mean; mIoU_std; mAcc_mean; mAcc_std
    Dezimaltrennzeichen ist Komma.
    Liefert Liste von Dikt-Records.
    """
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = [p.strip() for p in ln.split(';')]
            if len(parts) < 8:
                # ignoriere unvollständige Zeilen
                continue
            run = parts[0]
            dataset = parts[1]
            # konvertiere numerische Teile: Komma -> Punkt
            def to_float(s):
                s = s.replace(' ', '')
                s = s.replace(',', '.')
                try:
                    return float(s)
                except:
                    return None
            aAcc_mean = to_float(parts[2])
            aAcc_std  = to_float(parts[3])
            mIoU_mean = to_float(parts[4])
            mIoU_std  = to_float(parts[5])
            mAcc_mean = to_float(parts[6])
            mAcc_std  = to_float(parts[7])
            records.append({
                'run': run,
                'test_dataset': dataset,
                'aAcc_mean': aAcc_mean, 'aAcc_std': aAcc_std,
                'mIoU_mean': mIoU_mean, 'mIoU_std': mIoU_std,
                'mAcc_mean': mAcc_mean, 'mAcc_std': mAcc_std,
                'file': os.path.basename(path)
            })
    return records

def detect_preprocessing_from_run(run):
    """
    Erkanntes Mapping:
    bw -> grayscale
    co -> color-opponency
    sc -> single-color
    falls nichts gefunden: '-'
    """
    if re.search(r'_bw_', run):
        return 'grayscale'
    if re.search(r'_co_', run):
        return 'color-opponency'
    if re.search(r'_sc_', run):
        return 'single-color'
    return '-'

def detect_architecture_from_run(run):
    # Extrahiere Architektur aus run-String (heuristisch)
    m = re.search(r'train_([a-z0-9]+)', run)
    if m:
        return m.group(1)
    # fallback: ein Teil vor erstem underscore
    return run.split('_')[0]

def detect_sparsity_label_from_run(run, percent_based=False):
    """
    Für percent_based: z.B. '_sparse_percent_based_0,4' -> '40%'
    Für threshold/other: z.B. '_sparsity_0,0' -> '-'
    Wenn sparsity == 0 -> '-'
    """
    # versuche percent-based pattern
    m = re.search(r'sparse_percent_based_([0-9,\.]+)', run)
    if m:
        val = m.group(1).replace(',', '.')
        try:
            p = float(val) * 100.0
            # runde auf ganze Zahl wenn möglich
            if abs(p - round(p)) < 1e-6:
                p = int(round(p))
            else:
                p = round(p, 2)
            return f"{p}\\%"
        except:
            pass
    # sonst versuche sparsity pattern
    m2 = re.search(r'sparsity_([0-9,\.]+)', run)
    if m2:
        v = m2.group(1).replace(',', '.')
        try:
            f = float(v)
            if abs(f) < 1e-9:
                return '-'
            # interpretieren: wenn f between 0 and 1 and < 1 -> percent?
            return None
            """
            if 0 < f < 1:
                p = int(round(f*100))
                return f"{p}\\%"
            else:
                return str(f)"""
        except:
            pass
    # fallback
    return '-'

def fmt_mean_std(mean, std):
    if mean is None or std is None:
        return "0.00 ~$\\pm$~ 0.00"
    return f"{mean:.2f} ~$\\pm$~ {std:.2f}"

# ---------- Hauptlogik: Einlesen aller Dateien ----------
def load_all_results(data_dir):
    """
    Liest alle Dateien mit Namensmustern in data_dir (rekursiv nicht).
    Erwartet die drei Typen, aber ist robust gegenüber fehlenden Dateien.
    """
    patterns = [
        os.path.join(data_dir, "sparse_results_*_baseline_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*threshold_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*percent_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*_averaged_over_seed.txt"),  # allgemein fallback
    ]
    files = []
    for pat in patterns:
        files.extend(glob.glob(pat))
    files = sorted(set(files))
    all_records = []
    for fp in files:
        all_records += parse_result_lines(fp)
    return all_records

# ---------- Drehe die Rohdaten in strukturierte Form ----------
def build_table_records(records, target_arch, target_dataset, fill_missing_preproc_with_zeros=False):
    """
    Build table rows for LaTeX output.
    Produces:
      - first row: baseline '-' (no preprocessing)
      - then group of preprocessings with sparsity '-' (if exists)
      - then groups for each sparsity with multirow of preprocessings
    """
    # Filter dataset + architecture
    recs = [r for r in records if r['test_dataset'].lower() == target_dataset.lower()]
    matching = [r for r in recs if detect_architecture_from_run(r['run']).lower() == target_arch.lower()]

    # Map (preprocessing, sparsity) -> metrics
    table_map = {}
    preprocessings = set()
    sparsities = set()
    for r in matching:
        pre = detect_preprocessing_from_run(r['run'])
        sp = detect_sparsity_label_from_run(r['run'])

        if sp is None:
            continue

        table_map[(pre, sp)] = {
            'mIoU': (r['mIoU_mean'], r['mIoU_std']),
            'mAcc': (r['mAcc_mean'], r['mAcc_std']),
            'aAcc': (r['aAcc_mean'], r['aAcc_std']),
        }
        preprocessings.add(pre)
        sparsities.add(sp)

    # Fill missing preprocessings if desired
    if fill_missing_preproc_with_zeros and not preprocessings:
        for pre in ['grayscale', 'color-opponency', 'single-color']:
            preprocessings.add(pre)
            table_map[(pre, '-')] = {'mIoU': (0,0), 'mAcc': (0,0), 'aAcc': (0,0)}
        sparsities.add('-')

    # Sort sparsities: baseline '-' first
    def sparsity_sort_key(s):
        if s == '-': return -1
        m = re.match(r'([0-9]+)\%$', s)
        if m: return int(m.group(1))
        try: return float(s)
        except: return 1000
    sparsity_list = sorted(sparsities, key=sparsity_sort_key)

    # Sort preprocessings
    preproc_order = ['grayscale', 'color-opponency', 'single-color']
    preproc_list = [p for p in preproc_order if p in preprocessings] + [p for p in sorted(preprocessings) if p not in preproc_order]


    rows = []

    # ----------------
    # 1) Baseline single row: preprocessing='-', sparsity='-'
    # ----------------
    baseline_single_row = None
    if ('-', '-') in table_map:
        baseline_single_row = ('-', table_map[('-', '-')])
    else:
        # Fallback: erste Kombination mit sp='-' nehmen
        for (pre, sp), metrics in table_map.items():
            if sp == '-':
                baseline_single_row = (pre, metrics)
                break

    if baseline_single_row:
        pre, metrics = baseline_single_row
        rows.append({
            'architecture': target_arch,
            'preprocessing': '-',
            'sparsity': '-',
            'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
            'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
            'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
        })

    # ----------------
    # 2) Group of preprocessings for sparsity '-' (after baseline)
  
    # ----------------
    group_baseline = []
    for pre in preproc_list:
        if (pre, '-') in table_map:
            metrics = table_map[(pre, '-')]
        else:
            metrics = {'mIoU': (0.0,0.0), 'mAcc': (0.0,0.0), 'aAcc': (0.0,0.0)}
        group_baseline.append({
            'architecture': target_arch,
            'preprocessing': pre,
            'sparsity': '-',  # multirow with '-'
            'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
            'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
            'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
        })
    rows.extend(group_baseline)

    # ----------------
    # 3) Sparsity groups (alle außer '-')
    # ----------------
    for sp in sorted([s for s in sparsity_list if s != '-'], key=lambda x: int(re.match(r'([0-9]+)', x).group(1))):
        group_rows = []
        for pre in preproc_list:
            if (pre, sp) in table_map:
                metrics = table_map[(pre, sp)]
                group_rows.append({
                    'architecture': target_arch,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
                    'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
                    'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
                })
        rows.extend(group_rows)

    return rows



# ---------- Erzeuge LaTeX-Tabelle ----------
def rows_to_latex_table(rows, caption, label):
    """
    rows: Liste von dicts in der Reihenfolge, wie sie in der Tabelle erscheinen sollen.
    Erwartet: erste row kann baseline (sparsity='-'), danach Gruppen (preprocessing repeated for each sparsity group).
    Generiert LaTeX-String.
    """
    if not rows:
        return "% Keine Daten für diese Konfiguration."
    arch = rows[0]['architecture']
    # find baseline row (sparsity == '-'), assume first is baseline if present
    baseline = None
    baseline_group = []
    other_rows = []
    for r in rows:
        if r['sparsity'] == '-' and r['preprocessing'] == '-':
            baseline = r
        elif r['sparsity'] == '-' and r['preprocessing'] != '-':
            baseline_group.append(r)
        elif r['sparsity'] != '-' and r['preprocessing'] != '-':
            other_rows.append(r)
    # group other_rows by sparsity in order
    grouped = OrderedDict()
    for r in other_rows:
        grouped.setdefault(r['sparsity'], []).append(r)
    # build latex
    header = r"""\begin{table}[h]
  \centering
  \caption{%s}
  \begin{tabular}{@{}lccc ccc@{}}
    \toprule
    Architecture & Preprocessing & Sparsity & mIoU & mAcc & aAcc \\
    \midrule""" % caption
    lines = [header]
    # baseline
    if baseline:
        line = "    %s & - & - & %s & %s & %s \\\\" % (
            arch,
            fmt_mean_std(baseline['mIoU_mean'], baseline['mIoU_std']),
            fmt_mean_std(baseline['mAcc_mean'], baseline['mAcc_std']),
            fmt_mean_std(baseline['aAcc_mean'], baseline['aAcc_std'])
        )
        lines.append(line)
    lines.append("    \\midrule")
    lines.append("    \\midrule")
    if baseline_group:
        n = len(baseline_group)
        first = baseline_group[0]
        line = "    \\multirow{%d}{*}{%s} & %s & \\multirow{%d}{*}{-} & %s & %s & %s \\\\" % (
            n,
            arch,
            first['preprocessing'],
            n,
            fmt_mean_std(first['mIoU_mean'], first['mIoU_std']),
            fmt_mean_std(first['mAcc_mean'], first['mAcc_std']),
            fmt_mean_std(first['aAcc_mean'], first['aAcc_std'])
        )
        lines.append(line)
        for r in baseline_group[1:]:
            line = "    & %s & & %s & %s & %s \\\\" % (
                r['preprocessing'],
                fmt_mean_std(r['mIoU_mean'], r['mIoU_std']),
                fmt_mean_std(r['mAcc_mean'], r['mAcc_std']),
                fmt_mean_std(r['aAcc_mean'], r['aAcc_std'])
            )
            lines.append(line)
        lines.append("    \\midrule")

    # grouped sparsity blocks
    for sp, group in grouped.items():
        n = len(group)
        # first row: use multirow for architecture and sparsity
        first = group[0]
        line = "    \\multirow{%d}{*}{%s} & %s & \\multirow{%d}{*}{%s} & %s & %s & %s \\\\" % (
            n,
            arch,
            first['preprocessing'],
            n,
            sp,
            fmt_mean_std(first['mIoU_mean'], first['mIoU_std']),
            fmt_mean_std(first['mAcc_mean'], first['mAcc_std']),
            fmt_mean_std(first['aAcc_mean'], first['aAcc_std'])
        )
        lines.append(line)
        # remaining rows in this group
        for r in group[1:]:
            line = "    & %s & & %s & %s & %s \\\\" % (
                r['preprocessing'],
                fmt_mean_std(r['mIoU_mean'], r['mIoU_std']),
                fmt_mean_std(r['mAcc_mean'], r['mAcc_std']),
                fmt_mean_std(r['aAcc_mean'], r['aAcc_std'])
            )
            lines.append(line)
        lines.append("    \\midrule")
    footer = r"""    \bottomrule
  \end{tabular}
  \label{%s}
\end{table}""" % label
    lines.append(footer)
    return "\n".join(lines)

# ---------- Public helper: Erstellung einer Tabelle für gegebene Architektur und Test-Dataset ----------
def generate_latex_for(data_dir, architecture, test_dataset, caption=None, label=None, fill_missing=False):
    all_recs = load_all_results(data_dir)
    rows = build_table_records(all_recs, architecture, test_dataset, fill_missing_preproc_with_zeros=fill_missing)
    if caption is None:
        caption = f"Evaluation of the {architecture} architecture validated on {test_dataset} at different sparsities."
    if label is None:
        label = f"tab:sparse_results_{architecture}_{test_dataset}"
    latex = rows_to_latex_table(rows, caption, label)
    return latex

# ---------- Beispiel: wie man es benutzt ----------
if __name__ == "__main__":
    # Passe data_dir an dein Notebook / Dateiordner an:
    data_dir = "/home/lstracke/Visualization-Notebooks/trained_results"   # <-- hier deine txt-files ablegen

    # Architekturen
    architectures = ['mask2former', 'upernet', 'deeplabv3plus']
    # Test-Datasets
    test_datasets = ['cityscapes', 'dark_zurich', 'acdc_night', 'acdc_fog', 'acdc_rain', 'acdc_snow']

    for arch in architectures:
        for dataset in test_datasets:
            tex = generate_latex_for(
                data_dir,
                arch,
                dataset,
                caption=f"Evaluation of the {arch} architecture trained on Cityscapes and \\textbf{{validated on {dataset}}} at different sparsities.",
                label=f"tab:sparse_{arch}_{dataset}",
                fill_missing=(arch=='deeplabv3plus')  # fill zeros bei deeplabv3plus wenn gewünscht
            )
            out_path = f"./table_{arch}_{dataset}_percent.tex"
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(tex)
            print(f"Wrote {out_path}")


Wrote ./table_mask2former_cityscapes_percent.tex
Wrote ./table_mask2former_dark_zurich_percent.tex
Wrote ./table_mask2former_acdc_night_percent.tex
Wrote ./table_mask2former_acdc_fog_percent.tex
Wrote ./table_mask2former_acdc_rain_percent.tex
Wrote ./table_mask2former_acdc_snow_percent.tex
Wrote ./table_upernet_cityscapes_percent.tex
Wrote ./table_upernet_dark_zurich_percent.tex
Wrote ./table_upernet_acdc_night_percent.tex
Wrote ./table_upernet_acdc_fog_percent.tex
Wrote ./table_upernet_acdc_rain_percent.tex
Wrote ./table_upernet_acdc_snow_percent.tex
Wrote ./table_deeplabv3plus_cityscapes_percent.tex
Wrote ./table_deeplabv3plus_dark_zurich_percent.tex
Wrote ./table_deeplabv3plus_acdc_night_percent.tex
Wrote ./table_deeplabv3plus_acdc_fog_percent.tex
Wrote ./table_deeplabv3plus_acdc_rain_percent.tex
Wrote ./table_deeplabv3plus_acdc_snow_percent.tex


### threshold sparsity tables ###

In [ ]:
# Python 3 code für Jupyter Notebook
import os
import glob
import re
from collections import defaultdict, OrderedDict
import pandas as pd

# ---------- Hilfsfunktionen ----------
def parse_result_lines(path):
    """
    Erwartetes Format:
    run; test_dataset; aAcc_mean; aAcc_std; mIoU_mean; mIoU_std; mAcc_mean; mAcc_std
    Dezimaltrennzeichen ist Komma.
    Liefert Liste von Dikt-Records.
    """
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = [p.strip() for p in ln.split(';')]
            if len(parts) < 8:
                # ignoriere unvollständige Zeilen
                continue
            run = parts[0]
            dataset = parts[1]
            # konvertiere numerische Teile: Komma -> Punkt
            def to_float(s):
                s = s.replace(' ', '')
                s = s.replace(',', '.')
                try:
                    return float(s)
                except:
                    return None
            aAcc_mean = to_float(parts[2])
            aAcc_std  = to_float(parts[3])
            mIoU_mean = to_float(parts[4])
            mIoU_std  = to_float(parts[5])
            mAcc_mean = to_float(parts[6])
            mAcc_std  = to_float(parts[7])
            records.append({
                'run': run,
                'test_dataset': dataset,
                'aAcc_mean': aAcc_mean, 'aAcc_std': aAcc_std,
                'mIoU_mean': mIoU_mean, 'mIoU_std': mIoU_std,
                'mAcc_mean': mAcc_mean, 'mAcc_std': mAcc_std,
                'file': os.path.basename(path)
            })
    return records

def detect_preprocessing_from_run(run):
    """
    Erkanntes Mapping:
    bw -> grayscale
    co -> color-opponency
    sc -> single-color
    falls nichts gefunden: '-'
    """
    if re.search(r'_bw_', run):
        return 'grayscale'
    if re.search(r'_co_', run):
        return 'color-opponency'
    if re.search(r'_sc_', run):
        return 'single-color'
    return '-'

def detect_architecture_from_run(run):
    # Extrahiere Architektur aus run-String (heuristisch)
    m = re.search(r'train_([a-z0-9]+)', run)
    if m:
        return m.group(1)
    # fallback: ein Teil vor erstem underscore
    return run.split('_')[0]

def detect_sparsity_label_from_run(run, percent_based=False):
    """
    Für percent_based: z.B. '_sparse_percent_based_0,4' -> '40%'
    Für threshold/other: z.B. '_sparsity_0,0' -> '-'
    Wenn sparsity == 0 -> '-'
    """
    # versuche percent-based pattern
    m = re.search(r'sparse_percent_based_([0-9,\.]+)', run)
    if m:
        return None
        """
        val = m.group(1).replace(',', '.')
        try:
            p = float(val) * 100.0
            # runde auf ganze Zahl wenn möglich
            if abs(p - round(p)) < 1e-6:
                p = int(round(p))
            else:
                p = round(p, 2)
            return f"{p}\\%"
        except:
            pass"""
    # sonst versuche sparsity pattern
    m2 = re.search(r'sparsity_([0-9,\.]+)', run)
    if m2:
        v = m2.group(1).replace(',', '.')
        try:
            f = float(v)
            if abs(f) < 1e-6:
                return '-'
            elif abs(f) > 0.02:
                return None
            # interpretieren: wenn f between 0 and 1 and < 1 -> percent?
            else:
                return f"{f}\\%"
            return None
            """
            if 0 < f < 1:
                p = int(round(f*100))
                return f"{p}\\%"
            else:
                return str(f)"""
        except:
            pass
    # fallback
    return '-'

def fmt_mean_std(mean, std):
    if mean is None or std is None:
        return "0.00 ~$\\pm$~ 0.00"
    return f"{mean:.2f} ~$\\pm$~ {std:.2f}"

# ---------- Hauptlogik: Einlesen aller Dateien ----------
def load_all_results(data_dir):
    """
    Liest alle Dateien mit Namensmustern in data_dir (rekursiv nicht).
    Erwartet die drei Typen, aber ist robust gegenüber fehlenden Dateien.
    """
    patterns = [
        os.path.join(data_dir, "sparse_results_*_baseline_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*threshold_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*percent_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*_averaged_over_seed.txt"),  # allgemein fallback
    ]
    files = []
    for pat in patterns:
        files.extend(glob.glob(pat))
    files = sorted(set(files))
    all_records = []
    for fp in files:
        all_records += parse_result_lines(fp)
    return all_records

# ---------- Drehe die Rohdaten in strukturierte Form ----------
def build_table_records(records, target_arch, target_dataset, fill_missing_preproc_with_zeros=False):
    """
    Build table rows for LaTeX output.
    Produces:
      - first row: baseline '-' (no preprocessing)
      - then group of preprocessings with sparsity '-' (if exists)
      - then groups for each sparsity with multirow of preprocessings
    """
    # Filter dataset + architecture
    recs = [r for r in records if r['test_dataset'].lower() == target_dataset.lower()]
    matching = [r for r in recs if detect_architecture_from_run(r['run']).lower() == target_arch.lower()]

    # Map (preprocessing, sparsity) -> metrics
    table_map = {}
    preprocessings = set()
    sparsities = set()
    for r in matching:
        pre = detect_preprocessing_from_run(r['run'])
        sp = detect_sparsity_label_from_run(r['run'])

        
        if sp is None:
            continue
      
        table_map[(pre, sp)] = {
            'mIoU': (r['mIoU_mean'], r['mIoU_std']),
            'mAcc': (r['mAcc_mean'], r['mAcc_std']),
            'aAcc': (r['aAcc_mean'], r['aAcc_std']),
        }
        preprocessings.add(pre)
        sparsities.add(sp)

    # Fill missing preprocessings if desired
    if fill_missing_preproc_with_zeros and not preprocessings:
        for pre in ['grayscale', 'color-opponency', 'single-color']:
            preprocessings.add(pre)
            table_map[(pre, '-')] = {'mIoU': (0,0), 'mAcc': (0,0), 'aAcc': (0,0)}
        sparsities.add('-')

    # Sort sparsities: baseline '-' first
    def sparsity_sort_key(s):
        if s == '-': return -1
        m = re.match(r'([0-9]+)\%$', s)
        if m: return int(m.group(1))
        try: return float(s)
        except: return 1000
    sparsity_list = sorted(sparsities, key=sparsity_sort_key)

    # Sort preprocessings
    preproc_order = ['grayscale', 'color-opponency', 'single-color']
    preproc_list = [p for p in preproc_order if p in preprocessings] + [p for p in sorted(preprocessings) if p not in preproc_order]


    rows = []

    # ----------------
    # 1) Baseline single row: preprocessing='-', sparsity='-'
    # ----------------
    baseline_single_row = None
    if ('-', '-') in table_map:
        baseline_single_row = ('-', table_map[('-', '-')])
    else:
        # Fallback: erste Kombination mit sp='-' nehmen
        for (pre, sp), metrics in table_map.items():
            if sp == '-':
                baseline_single_row = (pre, metrics)
                break

    if baseline_single_row:
        pre, metrics = baseline_single_row
        rows.append({
            'architecture': target_arch,
            'preprocessing': '-',
            'sparsity': '-',
            'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
            'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
            'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
        })

    # ----------------
    # 2) Group of preprocessings for sparsity '-' (after baseline)
  
    # ----------------
    group_baseline = []
    for pre in preproc_list:
        if (pre, '-') in table_map:
            metrics = table_map[(pre, '-')]
        else:
            metrics = {'mIoU': (0.0,0.0), 'mAcc': (0.0,0.0), 'aAcc': (0.0,0.0)}
        group_baseline.append({
            'architecture': target_arch,
            'preprocessing': pre,
            'sparsity': '-',  # multirow with '-'
            'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
            'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
            'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
        })
    rows.extend(group_baseline)

    # ----------------
    # 3) Sparsity groups (alle außer '-')
    # ----------------
    for sp in sorted([s for s in sparsity_list if s != '-'], key=lambda x: int(re.match(r'([0-9]+)', x).group(1))):
        group_rows = []
        for pre in preproc_list:
            if (pre, sp) in table_map:
                metrics = table_map[(pre, sp)]
                group_rows.append({
                    'architecture': target_arch,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
                    'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
                    'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
                })
        rows.extend(group_rows)
    return rows



# ---------- Erzeuge LaTeX-Tabelle ----------
def rows_to_latex_table(rows, caption, label):
    """
    rows: Liste von dicts in der Reihenfolge, wie sie in der Tabelle erscheinen sollen.
    Erwartet: erste row kann baseline (sparsity='-'), danach Gruppen (preprocessing repeated for each sparsity group).
    Generiert LaTeX-String.
    """
    if not rows:
        return "% Keine Daten für diese Konfiguration."
    arch = rows[0]['architecture']
    # find baseline row (sparsity == '-'), assume first is baseline if present
    baseline = None
    baseline_group = []
    other_rows = []
    for r in rows:
        if r['sparsity'] == '-' and r['preprocessing'] == '-':
            baseline = r
        elif r['sparsity'] == '-' and r['preprocessing'] != '-':
            baseline_group.append(r)
        elif r['sparsity'] != '-' and r['preprocessing'] != '-':
            other_rows.append(r)
    # group other_rows by sparsity in order
    grouped = OrderedDict()
    for r in other_rows:
        grouped.setdefault(r['sparsity'], []).append(r)
    # build latex
    header = r"""\begin{table}[h]
  \centering
  \caption{%s}
  \begin{tabular}{@{}lccc ccc@{}}
    \toprule
    Architecture & Preprocessing & Sparsity & mIoU & mAcc & aAcc \\
    \midrule""" % caption
    lines = [header]
    # baseline
    if baseline:
        line = "    %s & - & - & %s & %s & %s \\\\" % (
            arch,
            fmt_mean_std(baseline['mIoU_mean'], baseline['mIoU_std']),
            fmt_mean_std(baseline['mAcc_mean'], baseline['mAcc_std']),
            fmt_mean_std(baseline['aAcc_mean'], baseline['aAcc_std'])
        )
        lines.append(line)
    lines.append("    \\midrule")
    lines.append("    \\midrule")
    if baseline_group:
        n = len(baseline_group)
        first = baseline_group[0]
        line = "    \\multirow{%d}{*}{%s} & %s & \\multirow{%d}{*}{-} & %s & %s & %s \\\\" % (
            n,
            arch,
            first['preprocessing'],
            n,
            fmt_mean_std(first['mIoU_mean'], first['mIoU_std']),
            fmt_mean_std(first['mAcc_mean'], first['mAcc_std']),
            fmt_mean_std(first['aAcc_mean'], first['aAcc_std'])
        )
        lines.append(line)
        for r in baseline_group[1:]:
            line = "    & %s & & %s & %s & %s \\\\" % (
                r['preprocessing'],
                fmt_mean_std(r['mIoU_mean'], r['mIoU_std']),
                fmt_mean_std(r['mAcc_mean'], r['mAcc_std']),
                fmt_mean_std(r['aAcc_mean'], r['aAcc_std'])
            )
            lines.append(line)
        lines.append("    \\midrule")

    # grouped sparsity blocks
    def sparsity_sort_key(s):
        if s == '-':
            return -1.0
        # Entferne % und \, wandle in float
        try:
            return float(s.replace('\\','').replace('%',''))
        except:
            return 1000.0

    # Erzeuge neues OrderedDict nach aufsteigender Sparsity
    grouped_sorted = OrderedDict(sorted(grouped.items(), key=lambda x: sparsity_sort_key(x[0])))
    for sp, group in grouped_sorted.items():
        n = len(group)
        # first row: use multirow for architecture and sparsity
        first = group[0]
        line = "    \\multirow{%d}{*}{%s} & %s & \\multirow{%d}{*}{%s} & %s & %s & %s \\\\" % (
            n,
            arch,
            first['preprocessing'],
            n,
            sp,
            fmt_mean_std(first['mIoU_mean'], first['mIoU_std']),
            fmt_mean_std(first['mAcc_mean'], first['mAcc_std']),
            fmt_mean_std(first['aAcc_mean'], first['aAcc_std'])
        )
        lines.append(line)
        # remaining rows in this group
        for r in group[1:]:
            line = "    & %s & & %s & %s & %s \\\\" % (
                r['preprocessing'],
                fmt_mean_std(r['mIoU_mean'], r['mIoU_std']),
                fmt_mean_std(r['mAcc_mean'], r['mAcc_std']),
                fmt_mean_std(r['aAcc_mean'], r['aAcc_std'])
            )
            lines.append(line)
        lines.append("    \\midrule")
    footer = r"""    \bottomrule
  \end{tabular}
  \label{%s}
\end{table}""" % label
    lines.append(footer)
    return "\n".join(lines)

# ---------- Public helper: Erstellung einer Tabelle für gegebene Architektur und Test-Dataset ----------
def generate_latex_for(data_dir, architecture, test_dataset, caption=None, label=None, fill_missing=False):
    all_recs = load_all_results(data_dir)
    rows = build_table_records(all_recs, architecture, test_dataset, fill_missing_preproc_with_zeros=fill_missing)
    if caption is None:
        caption = f"Evaluation of the {architecture} architecture validated on {test_dataset} at different sparsities."
    if label is None:
        label = f"tab:sparse_results_{architecture}_{test_dataset}"
    latex = rows_to_latex_table(rows, caption, label)
    return latex

# ---------- Beispiel: wie man es benutzt ----------
if __name__ == "__main__":
    # Passe data_dir an dein Notebook / Dateiordner an:
    data_dir = "/home/lstracke/Visualization-Notebooks/trained_results"   # <-- hier deine txt-files ablegen

    # Architekturen
    architectures = ['mask2former', 'upernet', 'deeplabv3plus']
    # Test-Datasets
    test_datasets = ['cityscapes', 'dark_zurich', 'acdc_night', 'acdc_fog', 'acdc_rain', 'acdc_snow']

    for arch in architectures:
        for dataset in test_datasets:
            tex = generate_latex_for(
                data_dir,
                arch,
                dataset,
                caption=f"Evaluation of the {arch} architecture trained on Cityscapes and \\textbf{{validated on {dataset}}} at different sparsities.",
                label=f"tab:sparse_{arch}_{dataset}",
                fill_missing=(arch=='deeplabv3plus')  # fill zeros bei deeplabv3plus wenn gewünscht
            )
            out_path = f"./table_{arch}_{dataset}_threshold.tex"
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(tex)
            print(f"Wrote {out_path}")



Wrote ./table_mask2former_cityscapes_threshold.tex
Wrote ./table_mask2former_dark_zurich_threshold.tex
Wrote ./table_mask2former_acdc_night_threshold.tex
Wrote ./table_mask2former_acdc_fog_threshold.tex
Wrote ./table_mask2former_acdc_rain_threshold.tex
Wrote ./table_mask2former_acdc_snow_threshold.tex
Wrote ./table_upernet_cityscapes_threshold.tex
Wrote ./table_upernet_dark_zurich_threshold.tex
Wrote ./table_upernet_acdc_night_threshold.tex
Wrote ./table_upernet_acdc_fog_threshold.tex
Wrote ./table_upernet_acdc_rain_threshold.tex
Wrote ./table_upernet_acdc_snow_threshold.tex
Wrote ./table_deeplabv3plus_cityscapes_threshold.tex
Wrote ./table_deeplabv3plus_dark_zurich_threshold.tex
Wrote ./table_deeplabv3plus_acdc_night_threshold.tex
Wrote ./table_deeplabv3plus_acdc_fog_threshold.tex
Wrote ./table_deeplabv3plus_acdc_rain_threshold.tex
Wrote ./table_deeplabv3plus_acdc_snow_threshold.tex


### Baseline tables ###

In [11]:
import os
import glob
import re

# --------------------------------------------------
# Parsing
# --------------------------------------------------

def parse_result_lines(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = [p.strip() for p in ln.split(';')]
            if len(parts) < 8:
                continue

            def to_float(s):
                s = s.replace(',', '.').replace(' ', '')
                try:
                    return float(s)
                except:
                    return None

            records.append({
                'run': parts[0],
                'dataset': parts[1],
                'aAcc_mean': to_float(parts[2]),
                'aAcc_std':  to_float(parts[3]),
                'mIoU_mean': to_float(parts[4]),
                'mIoU_std':  to_float(parts[5]),
                'mAcc_mean': to_float(parts[6]),
                'mAcc_std':  to_float(parts[7]),
            })
    return records


def load_all_results(data_dir):
    files = glob.glob(os.path.join(
        data_dir,
        "sparse_results_*_baseline_averaged_over_seed.txt"
    ))
    all_records = []
    for f in files:
        all_records += parse_result_lines(f)
    return all_records


# --------------------------------------------------
# Detection
# --------------------------------------------------

def architecture_matches(run, architecture):
    return f"train_{architecture}" in run


def detect_preprocessing(run):
    if '_bw_' in run:
        return 'grayscale'
    if '_co_' in run:
        return 'color-opponency'
    if '_sc_' in run:
        return 'single-color'
    return '-'


def extract_sparsity(run):
    m = re.search(r'sparsity_([0-9,\.]+)', run)
    if not m:
        return None
    return float(m.group(1).replace(',', '.'))


def fmt(mean, std):
    if mean is None or std is None:
        return "0.00 ~$\\pm$~ 0.00"
    return f"{mean:.2f} ~$\\pm$~ {std:.2f}"


# --------------------------------------------------
# Build grouped rows
# --------------------------------------------------

def build_grouped_rows(records, architecture, datasets,
                       percent_values=[40,50,60,70,80,90]):

    grouped = []

    for ds in datasets:

        recs = [
            r for r in records
            if r['dataset'].lower() == ds.lower()
            and architecture_matches(r['run'], architecture)
            and detect_preprocessing(r['run']) == '-'
        ]

        sparsity_map = {}

        for r in recs:
            val = extract_sparsity(r['run'])
            if val is None:
                continue
            sparsity_map[val] = r

        dataset_block = []

        # ---- Baseline 0.0 ----
        if 0.0 in sparsity_map:
            r = sparsity_map[0.0]
            dataset_block.append({
                'architecture': architecture,
                'dataset': ds,
                'sparsity': '-',
                'mIoU': fmt(r['mIoU_mean'], r['mIoU_std']),
                'mAcc': fmt(r['mAcc_mean'], r['mAcc_std']),
                'aAcc': fmt(r['aAcc_mean'], r['aAcc_std'])
            })

        # ---- Prozent 40–90 ----
        for pv in percent_values:
            val = pv / 100.0
            if val in sparsity_map:
                r = sparsity_map[val]
                dataset_block.append({
                    'architecture': architecture,
                    'dataset': ds,
                    'sparsity': f"{pv}\\%",
                    'mIoU': fmt(r['mIoU_mean'], r['mIoU_std']),
                    'mAcc': fmt(r['mAcc_mean'], r['mAcc_std']),
                    'aAcc': fmt(r['aAcc_mean'], r['aAcc_std'])
                })

        grouped.append(dataset_block)

    return grouped


# --------------------------------------------------
# LaTeX Generator
# --------------------------------------------------

def generate_latex(grouped_rows, caption, label):

    header = r"""\begin{table}[h]
  \centering
  \caption{%s}
  \begin{tabular}{@{}l l c c c c@{}}
    \toprule
    Architecture & Dataset & Sparsity & mIoU & mAcc & aAcc \\
    \midrule""" % caption

    lines = [header]
    first_block = True

    for block in grouped_rows:
        if not block:
            continue

        if not first_block:
            lines.append("    \\midrule")
        first_block = False

        n = len(block)
        arch = block[0]['architecture']
        ds = block[0]['dataset']

        r0 = block[0]

        lines.append(
            f"    \\multirow{{{n}}}{{*}}{{{arch}}} & "
            f"\\multirow{{{n}}}{{*}}{{{ds}}} & "
            f"{r0['sparsity']} & {r0['mIoU']} & {r0['mAcc']} & {r0['aAcc']} \\\\"
        )

        for r in block[1:]:
            lines.append(
                f"    & & {r['sparsity']} & {r['mIoU']} & {r['mAcc']} & {r['aAcc']} \\\\"
            )

    footer = r"""    \bottomrule
  \end{tabular}
  \label{%s}
\end{table}""" % label

    lines.append(footer)
    return "\n".join(lines)


# --------------------------------------------------
# AUSFÜHRUNG
# --------------------------------------------------

if __name__ == "__main__":

    data_dir = "/home/lstracke/Visualization-Notebooks/trained_results"

    datasets = [
        'cityscapes',
        'dark_zurich',
        'acdc_night',
        'acdc_fog',
        'acdc_rain',
        'acdc_snow'
    ]

    architectures = ['mask2former', 'upernet', 'deeplabv3plus']

    records = load_all_results(data_dir)

    for arch in architectures:

        grouped = build_grouped_rows(records, arch, datasets)

        latex = generate_latex(
            grouped,
            caption=f"Baseline (0\\%) and sparsities 40--90\\% for {arch}.",
            label=f"tab:sparsity_{arch}"
        )

        with open(f"./table_baseline_percentage_{arch}_grouped.tex", "w") as f:
            f.write(latex)

        print(f"Wrote table_baseline_percentage_{arch}_grouped.tex")


Wrote table_baseline_percentage_mask2former_grouped.tex
Wrote table_baseline_percentage_upernet_grouped.tex
Wrote table_baseline_percentage_deeplabv3plus_grouped.tex


### Combination of baseline with percentages ###

In [ ]:
# combined_table_generator.py
# Python 3 - erzeugt LaTeX-Tabellen: pro Dataset + Architektur eigene Tabelle
import os
import glob
import re
from itertools import groupby

# ---------------------------
# Einlesen / Parsen
# ---------------------------
def parse_result_lines(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = [p.strip() for p in ln.split(';')]
            if len(parts) < 8:
                continue
            def to_float(s):
                if s is None:
                    return None
                s = s.replace(' ', '').replace(',', '.')
                try:
                    return float(s)
                except:
                    return None
            records.append({
                'run': parts[0],
                'dataset': parts[1],
                'aAcc_mean': to_float(parts[2]),
                'aAcc_std':  to_float(parts[3]),
                'mIoU_mean': to_float(parts[4]),
                'mIoU_std':  to_float(parts[5]),
                'mAcc_mean': to_float(parts[6]),
                'mAcc_std':  to_float(parts[7]),
                'file': os.path.basename(path)
            })
    return records

def load_all_results(data_dir):
    patterns = [
        os.path.join(data_dir, "sparse_results_*_baseline_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*threshold_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*percent_averaged_over_seed.txt"),
        os.path.join(data_dir, "sparse_results_*_averaged_over_seed.txt"),
    ]
    files = []
    for p in patterns:
        files.extend(glob.glob(p))
    files = sorted(set(files))
    all_records = []
    for fp in files:
        all_records += parse_result_lines(fp)
    return all_records

# ---------------------------
# Erkennung
# ---------------------------
def architecture_matches(run, arch):
    return f"train_{arch}" in run

def detect_preprocessing(run):
    if re.search(r'_bw_', run):
        return 'grayscale'
    if re.search(r'_co_', run):
        return 'color-opponency'
    if re.search(r'_sc_', run):
        return 'single-color'
    return '-'

def detect_sparsity_label(run):
    m = re.search(r'sparse_percent_based_([0-9,\.]+)', run)
    if m:
        try:
            v = float(m.group(1).replace(',', '.'))
            p = int(round(v * 100)) if v < 1 else int(round(v))
            return f"{p}\\%"
        except:
            pass
    m2 = re.search(r'percent[_-]([0-9,\.]+)', run)
    if m2:
        try:
            v = float(m2.group(1).replace(',', '.'))
            p = int(round(v * 100)) if v < 1 else int(round(v))
            return f"{p}\\%"
        except:
            pass
    m3 = re.search(r'sparsity_([0-9,\.]+)', run)
    if m3:
        try:
            v = float(m3.group(1).replace(',', '.'))
            if abs(v) < 1e-9:
                return '-'
            if 0 < v < 1:
                p = int(round(v * 100))
                return f"{p}\\%"
            if v >= 1:
                return f"{int(round(v))}\\%"
        except:
            pass
    m4 = re.search(r'_sparse_([0-9,\.]+)', run)
    if m4:
        try:
            v = float(m4.group(1).replace(',', '.'))
            if abs(v) < 1e-9:
                return '-'
            if 0 < v < 1:
                p = int(round(v*100))
                return f"{p}\\%"
            if v >= 1:
                return f"{int(round(v))}\\%"
        except:
            pass
    return None

def fmt_mean_std(mean, std):
    if mean is None or std is None:
        return "0.00 ~$\\pm$~ 0.00"
    return f"{mean:.2f}~$\\pm$~{std:.2f}"

# ---------------------------
# Tabellen-Aufbau
# ---------------------------
def build_full_table_map(records, architecture, dataset, fill_missing=False):
    recs = [r for r in records if r['dataset'].lower() == dataset.lower() and architecture_matches(r['run'], architecture)]
    mp = {}
    sparsities = set()
    preprocessings = set(['-', 'grayscale', 'color-opponency', 'single-color'])
    for r in recs:
        pre = detect_preprocessing(r['run'])
        sp = detect_sparsity_label(r['run'])
        if sp is None:
            continue
        mp[(sp, pre)] = r
        sparsities.add(sp)
    if not sparsities and fill_missing:
        sparsities.add('-')
        for pre in preprocessings:
            mp[('-', pre)] = None

    def sparsity_key(s):
        if s == '-': return -1
        m = re.match(r'([0-9]+)', s)
        if m:
            return int(m.group(1))
        return 1000
    sparsity_list = sorted(sparsities, key=sparsity_key)
    pre_order = ['-', 'grayscale', 'color-opponency', 'single-color']

    rows = []
    for sp in sparsity_list:
        for pre in pre_order:
            rec = mp.get((sp, pre))
            if rec is None and not fill_missing:
                continue
            if rec is None:
                rows.append({
                    'architecture': architecture,
                    'dataset': dataset,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU': fmt_mean_std(None, None),
                    'mAcc': fmt_mean_std(None, None),
                    'aAcc': fmt_mean_std(None, None)
                })
            else:
                rows.append({
                    'architecture': architecture,
                    'dataset': dataset,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU': fmt_mean_std(rec['mIoU_mean'], rec['mIoU_std']),
                    'mAcc': fmt_mean_std(rec['mAcc_mean'], rec['mAcc_std']),
                    'aAcc': fmt_mean_std(rec['aAcc_mean'], rec['aAcc_std'])
                })
    return rows

# ---------------------------
# LaTeX-Generator mit Sparsity-Multirow
# ---------------------------

ARCH_NAME = {
    'mask2former': 'Mask2Former',
    'upernet': 'UPerNet',
    'deeplabv3plus': 'DeepLabv3+'
}

PRE_NAME = {
    '-': '-',
    'grayscale': 'Luminance',
    'single-color': 'Single-color',
    'color-opponency': 'Color-opponency'
}

def rows_to_latex(rows, caption, label, hide_dataset=True):
    if not rows:
        return "% Keine Daten für diese Konfiguration."

    header = r"""\begin{table}[h]
  \centering
  \caption{%s}
  \begin{tabular}{@{}l l c c c c@{}}
    \toprule
    Architecture & Preprocessing & Sparsity & mIoU & mAcc & aAcc \\
    \midrule""" % caption
    lines = [header]

    rows_sorted = sorted(rows, key=lambda r: (r['architecture'], r['sparsity']))
    for arch, arch_grp in groupby(rows_sorted, key=lambda r: r['architecture']):
        arch_list = list(arch_grp)
        # gruppiere nach Sparsity
        sparsity_groups = {}
        for r in arch_list:
            sparsity_groups.setdefault(r['sparsity'], []).append(r)
        for sp, grp in sparsity_groups.items():
            n = len(grp)
            r0 = grp[0]
            lines.append(
                f"    \\multirow{{{n}}}{{*}}{{{ARCH_NAME.get(arch, arch)}}} & "
                f"{PRE_NAME.get(r0['preprocessing'], r0['preprocessing'])} & "
                f"\\multirow{{{n}}}{{*}}{{{sp}}} & "
                f"{r0['mIoU']} & {r0['mAcc']} & {r0['aAcc']} \\\\"
            )
            for r in grp[1:]:
                lines.append(
                    f"    & {PRE_NAME.get(r['preprocessing'], r['preprocessing'])} & "
                    f" & {r['mIoU']} & {r['mAcc']} & {r['aAcc']} \\\\"
                )
            lines.append("    \\midrule")

    footer = r"""    \bottomrule
  \end{tabular}
  \label{%s}
\end{table}""" % label
    lines.append(footer)
    return "\n".join(lines)

# ---------------------------
# Erzeuge pro Dataset + Architektur .tex
# ---------------------------
def generate_tables_per_dataset(data_dir, architectures, datasets, out_dir=".", fill_missing=True):
    os.makedirs(out_dir, exist_ok=True)
    records = load_all_results(data_dir)
    for arch in architectures:
        for ds in datasets:
            rows = build_full_table_map(records, arch, ds, fill_missing=fill_missing)
            # Dataset-Spalte entfernen
            for r in rows:
                r.pop('dataset', None)

            caption = f"For {arch} on {ds}: all sparsities (incl. '-') and preprocessings"
            label = f"tab:{arch}_{ds}"
            latex = rows_to_latex(rows, caption, label, hide_dataset=True)
            out_path = os.path.join(out_dir, f"table_{arch}_{ds}_combined.tex")
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(latex)
            print(f"Wrote {out_path}")

# ---------------------------
# Beispiel-Aufruf
# ---------------------------
if __name__ == "__main__":
    data_dir = "/home/lstracke/Visualization-Notebooks/trained_results"  # anpassen
    architectures = ['mask2former', 'upernet', 'deeplabv3plus']
    datasets = ['cityscapes', 'dark_zurich', 'acdc_night', 'acdc_fog', 'acdc_rain', 'acdc_snow']
    generate_tables_per_dataset(data_dir, architectures, datasets, out_dir=".", fill_missing=True)


Wrote ./table_mask2former_cityscapes_combined.tex
Wrote ./table_mask2former_dark_zurich_combined.tex
Wrote ./table_mask2former_acdc_night_combined.tex
Wrote ./table_mask2former_acdc_fog_combined.tex
Wrote ./table_mask2former_acdc_rain_combined.tex
Wrote ./table_mask2former_acdc_snow_combined.tex
Wrote ./table_upernet_cityscapes_combined.tex
Wrote ./table_upernet_dark_zurich_combined.tex
Wrote ./table_upernet_acdc_night_combined.tex
Wrote ./table_upernet_acdc_fog_combined.tex
Wrote ./table_upernet_acdc_rain_combined.tex
Wrote ./table_upernet_acdc_snow_combined.tex
Wrote ./table_deeplabv3plus_cityscapes_combined.tex
Wrote ./table_deeplabv3plus_dark_zurich_combined.tex
Wrote ./table_deeplabv3plus_acdc_night_combined.tex
Wrote ./table_deeplabv3plus_acdc_fog_combined.tex
Wrote ./table_deeplabv3plus_acdc_rain_combined.tex
Wrote ./table_deeplabv3plus_acdc_snow_combined.tex


### Percent results learned with acdc_full ###

In [10]:
# combined_table_generator.py
# Python 3 - erzeugt LaTeX-Tabellen: pro Dataset + Architektur eigene Tabelle
import os
import glob
import re
from itertools import groupby

# ---------------------------
# Einlesen / Parsen
# ---------------------------
def parse_result_lines(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = [p.strip() for p in ln.split(';')]
            if len(parts) < 8:
                continue
            def to_float(s):
                if s is None:
                    return None
                s = s.replace(' ', '').replace(',', '.')
                try:
                    return float(s)
                except:
                    return None
            records.append({
                'run': parts[0],
                'dataset': parts[1],
                'aAcc_mean': to_float(parts[2]),
                'aAcc_std':  to_float(parts[3]),
                'mIoU_mean': to_float(parts[4]),
                'mIoU_std':  to_float(parts[5]),
                'mAcc_mean': to_float(parts[6]),
                'mAcc_std':  to_float(parts[7]),
                'file': os.path.basename(path)
            })
    return records

def load_all_results(data_dir):
    patterns = [
        os.path.join(data_dir, "results_with_acdc_full/sparse_results_*_baseline_acdc_full_averaged_over_seed.txt"),
        os.path.join(data_dir, "results_with_acdc_full/sparse_results_*threshold_acdc_full_averaged_over_seed.txt"),
        os.path.join(data_dir, "results_with_acdc_full/sparse_results_*percent_acdc_full_averaged_over_seed.txt")
    ]
    files = []
    for p in patterns:
        files.extend(glob.glob(p))
    files = sorted(set(files))
    all_records = []
    for fp in files:
        all_records += parse_result_lines(fp)
    return all_records

# ---------------------------
# Erkennung
# ---------------------------
def architecture_matches(run, arch):
    return f"train_{arch}" in run

def detect_preprocessing(run):
    if re.search(r'_bw_', run):
        return 'grayscale'
    if re.search(r'_co_', run):
        return 'color-opponency'
    if re.search(r'_sc_', run):
        return 'single-color'
    return '-'

def detect_sparsity_label(run):
    m = re.search(r'sparse_percent_based_([0-9,\.]+)', run)
    if m:
        try:
            v = float(m.group(1).replace(',', '.'))
            p = int(round(v * 100)) if v < 1 else int(round(v))
            return f"{p}\\%"
        except:
            pass
    m2 = re.search(r'percent[_-]([0-9,\.]+)', run)
    if m2:
        try:
            v = float(m2.group(1).replace(',', '.'))
            p = int(round(v * 100)) if v < 1 else int(round(v))
            return f"{p}\\%"
        except:
            pass
    m3 = re.search(r'sparsity_([0-9,\.]+)', run)
    if m3:
        try:
            v = float(m3.group(1).replace(',', '.'))
            if abs(v) < 1e-9:
                return '-'
            if 0 < v < 1:
                p = int(round(v * 100))
                return f"{p}\\%"
            if v >= 1:
                return f"{int(round(v))}\\%"
        except:
            pass
    m4 = re.search(r'_sparse_([0-9,\.]+)', run)
    if m4:
        try:
            v = float(m4.group(1).replace(',', '.'))
            if abs(v) < 1e-9:
                return '-'
            if 0 < v < 1:
                p = int(round(v*100))
                return f"{p}\\%"
            if v >= 1:
                return f"{int(round(v))}\\%"
        except:
            pass
    return None

def fmt_mean_std(mean, std):
    if mean is None or std is None:
        return "0.00 ~$\\pm$~ 0.00"
    return f"{mean:.2f}~$\\pm$~{std:.2f}"

# ---------------------------
# Tabellen-Aufbau
# ---------------------------
def build_full_table_map(records, architecture, dataset, fill_missing=False):
    recs = [r for r in records if r['dataset'].lower() == dataset.lower() and architecture_matches(r['run'], architecture)]
    mp = {}
    sparsities = set()
    preprocessings = set(['-', 'grayscale', 'color-opponency', 'single-color'])
    for r in recs:
        pre = detect_preprocessing(r['run'])
        sp = detect_sparsity_label(r['run'])
        if sp is None:
            continue
        mp[(sp, pre)] = r
        sparsities.add(sp)
    if not sparsities and fill_missing:
        sparsities.add('-')
        for pre in preprocessings:
            mp[('-', pre)] = None

    def sparsity_key(s):
        if s == '-': return -1
        m = re.match(r'([0-9]+)', s)
        if m:
            return int(m.group(1))
        return 1000
    sparsity_list = sorted(sparsities, key=sparsity_key)
    pre_order = ['-', 'grayscale', 'color-opponency', 'single-color']

    rows = []
    for sp in sparsity_list:
        for pre in pre_order:
            rec = mp.get((sp, pre))
            if rec is None and not fill_missing:
                continue
            if rec is None:
                rows.append({
                    'architecture': architecture,
                    'dataset': dataset,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU': fmt_mean_std(None, None),
                    'mAcc': fmt_mean_std(None, None),
                    'aAcc': fmt_mean_std(None, None)
                })
            else:
                rows.append({
                    'architecture': architecture,
                    'dataset': dataset,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU': fmt_mean_std(rec['mIoU_mean'], rec['mIoU_std']),
                    'mAcc': fmt_mean_std(rec['mAcc_mean'], rec['mAcc_std']),
                    'aAcc': fmt_mean_std(rec['aAcc_mean'], rec['aAcc_std'])
                })
    return rows

# ---------------------------
# LaTeX-Generator mit Sparsity-Multirow
# ---------------------------

ARCH_NAME = {
    'mask2former': 'Mask2Former',
    'upernet': 'UPerNet',
    'deeplabv3plus': 'DeepLabv3+'
}

PRE_NAME = {
    '-': '-',
    'grayscale': 'Luminance',
    'single-color': 'Single-color',
    'color-opponency': 'Color-opponency'
}

def rows_to_latex(rows, caption, label, hide_dataset=True):
    if not rows:
        return "% Keine Daten für diese Konfiguration."

    header = r"""\begin{table}[h]
  \centering
  \caption{%s}
  \begin{tabular}{@{}l l c c c c@{}}
    \toprule
    Architecture & Preprocessing & Sparsity & mIoU & mAcc & aAcc \\
    \midrule""" % caption
    lines = [header]

    rows_sorted = sorted(rows, key=lambda r: (r['architecture'], r['sparsity']))
    for arch, arch_grp in groupby(rows_sorted, key=lambda r: r['architecture']):
        arch_list = list(arch_grp)
        # gruppiere nach Sparsity
        sparsity_groups = {}
        for r in arch_list:
            sparsity_groups.setdefault(r['sparsity'], []).append(r)
        for sp, grp in sparsity_groups.items():
            n = len(grp)
            r0 = grp[0]
            lines.append(
                f"    \\multirow{{{n}}}{{*}}{{{ARCH_NAME.get(arch, arch)}}} & "
                f"{PRE_NAME.get(r0['preprocessing'], r0['preprocessing'])} & "
                f"\\multirow{{{n}}}{{*}}{{{sp}}} & "
                f"{r0['mIoU']} & {r0['mAcc']} & {r0['aAcc']} \\\\"
            )
            for r in grp[1:]:
                lines.append(
                    f"    & {PRE_NAME.get(r['preprocessing'], r['preprocessing'])} & "
                    f" & {r['mIoU']} & {r['mAcc']} & {r['aAcc']} \\\\"
                )
            lines.append("    \\midrule")

    footer = r"""    \bottomrule
  \end{tabular}
  \label{%s}
\end{table}""" % label
    lines.append(footer)
    return "\n".join(lines)

# ---------------------------
# Erzeuge pro Dataset + Architektur .tex
# ---------------------------
def generate_tables_per_dataset(data_dir, architectures, datasets, out_dir=".", fill_missing=True):
    os.makedirs(out_dir, exist_ok=True)
    records = load_all_results(data_dir)
    for arch in architectures:
        for ds in datasets:
            rows = build_full_table_map(records, arch, ds, fill_missing=fill_missing)
            # Dataset-Spalte entfernen
            for r in rows:
                r.pop('dataset', None)

            caption = f"For {arch} on {ds}: all sparsities (incl. '-') and preprocessings"
            label = f"tab:{arch}_{ds}"
            latex = rows_to_latex(rows, caption, label, hide_dataset=True)
            out_path = os.path.join(out_dir, f"table_{arch}_{ds}_combined.tex")
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(latex)
            print(f"Wrote {out_path}")

# ---------------------------
# Beispiel-Aufruf
# ---------------------------
if __name__ == "__main__":
    data_dir = "/home/lstracke/Visualization-Notebooks/trained_results"  # anpassen
    architectures = ['mask2former', 'upernet', 'deeplabv3plus']
    datasets = ['acdc_full'] #'cityscapes', 'dark_zurich', 'acdc_night', 'acdc_fog', 'acdc_rain', 'acdc_snow']
    generate_tables_per_dataset(data_dir, architectures, datasets, out_dir=".", fill_missing=True)


Wrote ./table_mask2former_acdc_full_combined.tex
Wrote ./table_upernet_acdc_full_combined.tex
Wrote ./table_deeplabv3plus_acdc_full_combined.tex


### ACDC_full threshold tables ###

In [12]:
# Python 3 code für Jupyter Notebook
import os
import glob
import re
from collections import defaultdict, OrderedDict
import pandas as pd

# ---------- Hilfsfunktionen ----------
def parse_result_lines(path):
    """
    Erwartetes Format:
    run; test_dataset; aAcc_mean; aAcc_std; mIoU_mean; mIoU_std; mAcc_mean; mAcc_std
    Dezimaltrennzeichen ist Komma.
    Liefert Liste von Dikt-Records.
    """
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = [p.strip() for p in ln.split(';')]
            if len(parts) < 8:
                # ignoriere unvollständige Zeilen
                continue
            run = parts[0]
            dataset = parts[1]
            # konvertiere numerische Teile: Komma -> Punkt
            def to_float(s):
                s = s.replace(' ', '')
                s = s.replace(',', '.')
                try:
                    return float(s)
                except:
                    return None
            aAcc_mean = to_float(parts[2])
            aAcc_std  = to_float(parts[3])
            mIoU_mean = to_float(parts[4])
            mIoU_std  = to_float(parts[5])
            mAcc_mean = to_float(parts[6])
            mAcc_std  = to_float(parts[7])
            records.append({
                'run': run,
                'test_dataset': dataset,
                'aAcc_mean': aAcc_mean, 'aAcc_std': aAcc_std,
                'mIoU_mean': mIoU_mean, 'mIoU_std': mIoU_std,
                'mAcc_mean': mAcc_mean, 'mAcc_std': mAcc_std,
                'file': os.path.basename(path)
            })
    return records

def detect_preprocessing_from_run(run):
    """
    Erkanntes Mapping:
    bw -> grayscale
    co -> color-opponency
    sc -> single-color
    falls nichts gefunden: '-'
    """
    if re.search(r'_bw_', run):
        return 'grayscale'
    if re.search(r'_co_', run):
        return 'color-opponency'
    if re.search(r'_sc_', run):
        return 'single-color'
    return '-'

def detect_architecture_from_run(run):
    # Extrahiere Architektur aus run-String (heuristisch)
    m = re.search(r'train_([a-z0-9]+)', run)
    if m:
        return m.group(1)
    # fallback: ein Teil vor erstem underscore
    return run.split('_')[0]

def detect_sparsity_label_from_run(run, percent_based=False):
    """
    Für percent_based: z.B. '_sparse_percent_based_0,4' -> '40%'
    Für threshold/other: z.B. '_sparsity_0,0' -> '-'
    Wenn sparsity == 0 -> '-'
    """
    # versuche percent-based pattern
    m = re.search(r'sparse_percent_based_([0-9,\.]+)', run)
    if m:
        return None
        """
        val = m.group(1).replace(',', '.')
        try:
            p = float(val) * 100.0
            # runde auf ganze Zahl wenn möglich
            if abs(p - round(p)) < 1e-6:
                p = int(round(p))
            else:
                p = round(p, 2)
            return f"{p}\\%"
        except:
            pass"""
    # sonst versuche sparsity pattern
    m2 = re.search(r'sparsity_([0-9,\.]+)', run)
    if m2:
        v = m2.group(1).replace(',', '.')
        try:
            f = float(v)
            if abs(f) < 1e-6:
                return '-'
            elif abs(f) > 0.02:
                return None
            # interpretieren: wenn f between 0 and 1 and < 1 -> percent?
            else:
                return f"{f}\\%"
            return None
            """
            if 0 < f < 1:
                p = int(round(f*100))
                return f"{p}\\%"
            else:
                return str(f)"""
        except:
            pass
    # fallback
    return '-'

def fmt_mean_std(mean, std):
    if mean is None or std is None:
        return "0.00 ~$\\pm$~ 0.00"
    return f"{mean:.2f} ~$\\pm$~ {std:.2f}"

# ---------- Name Mapping für LaTeX ----------
ARCH_NAME = {
    'mask2former': 'Mask2Former',
    'upernet': 'UPerNet',
    'deeplabv3plus': 'DeepLabv3+'
}

PRE_NAME = {
    '-': '-',
    'grayscale': 'Luminance',
    'single-color': 'Single-color',
    'color-opponency': 'Color-opponency'
}


# ---------- Hauptlogik: Einlesen aller Dateien ----------
def load_all_results(data_dir):
    """
    Liest alle Dateien mit Namensmustern in data_dir (rekursiv nicht).
    Erwartet die drei Typen, aber ist robust gegenüber fehlenden Dateien.
    """

    
    patterns = [
        os.path.join(data_dir, "results_with_acdc_full/sparse_results_*_baseline_acdc_full_averaged_over_seed.txt"),
        os.path.join(data_dir, "results_with_acdc_full/sparse_results_*threshold_acdc_full_averaged_over_seed.txt"),
        os.path.join(data_dir, "results_with_acdc_full/sparse_results_*percent_acdc_full_averaged_over_seed.txt")
    ]
    files = []
    for pat in patterns:
        files.extend(glob.glob(pat))
    files = sorted(set(files))
    all_records = []
    for fp in files:
        all_records += parse_result_lines(fp)
    return all_records

# ---------- Drehe die Rohdaten in strukturierte Form ----------
def build_table_records(records, target_arch, target_dataset, fill_missing_preproc_with_zeros=False):
    """
    Build table rows for LaTeX output.
    Produces:
      - first row: baseline '-' (no preprocessing)
      - then group of preprocessings with sparsity '-' (if exists)
      - then groups for each sparsity with multirow of preprocessings
    """
    # Filter dataset + architecture
    recs = [r for r in records if r['test_dataset'].lower() == target_dataset.lower()]
    matching = [r for r in recs if detect_architecture_from_run(r['run']).lower() == target_arch.lower()]

    # Map (preprocessing, sparsity) -> metrics
    table_map = {}
    preprocessings = set()
    sparsities = set()
    for r in matching:
        pre = detect_preprocessing_from_run(r['run'])
        sp = detect_sparsity_label_from_run(r['run'])

        
        if sp is None:
            continue
      
        table_map[(pre, sp)] = {
            'mIoU': (r['mIoU_mean'], r['mIoU_std']),
            'mAcc': (r['mAcc_mean'], r['mAcc_std']),
            'aAcc': (r['aAcc_mean'], r['aAcc_std']),
        }
        preprocessings.add(pre)
        sparsities.add(sp)

    # Fill missing preprocessings if desired
    if fill_missing_preproc_with_zeros and not preprocessings:
        for pre in ['grayscale', 'color-opponency', 'single-color']:
            preprocessings.add(pre)
            table_map[(pre, '-')] = {'mIoU': (0,0), 'mAcc': (0,0), 'aAcc': (0,0)}
        sparsities.add('-')

    # Sort sparsities: baseline '-' first
    def sparsity_sort_key(s):
        if s == '-': return -1
        m = re.match(r'([0-9]+)\%$', s)
        if m: return int(m.group(1))
        try: return float(s)
        except: return 1000
    sparsity_list = sorted(sparsities, key=sparsity_sort_key)

    # Sort preprocessings
    preproc_order = ['grayscale', 'color-opponency', 'single-color']
    preproc_list = [p for p in preproc_order if p in preprocessings] + [p for p in sorted(preprocessings) if p not in preproc_order]


    rows = []

    # ----------------
    # 1) Baseline single row: preprocessing='-', sparsity='-'
    # ----------------
    baseline_single_row = None
    if ('-', '-') in table_map:
        baseline_single_row = ('-', table_map[('-', '-')])
    else:
        # Fallback: erste Kombination mit sp='-' nehmen
        for (pre, sp), metrics in table_map.items():
            if sp == '-':
                baseline_single_row = (pre, metrics)
                break

    if baseline_single_row:
        pre, metrics = baseline_single_row
        rows.append({
            'architecture': target_arch,
            'preprocessing': '-',
            'sparsity': '-',
            'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
            'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
            'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
        })

    # ----------------
    # 2) Group of preprocessings for sparsity '-' (after baseline)
  
    # ----------------
    group_baseline = []
    for pre in preproc_list:
        if (pre, '-') in table_map:
            metrics = table_map[(pre, '-')]
        else:
            metrics = {'mIoU': (0.0,0.0), 'mAcc': (0.0,0.0), 'aAcc': (0.0,0.0)}
        group_baseline.append({
            'architecture': target_arch,
            'preprocessing': pre,
            'sparsity': '-',  # multirow with '-'
            'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
            'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
            'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
        })
    rows.extend(group_baseline)

    # ----------------
    # 3) Sparsity groups (alle außer '-')
    # ----------------
    for sp in sorted([s for s in sparsity_list if s != '-'], key=lambda x: int(re.match(r'([0-9]+)', x).group(1))):
        group_rows = []
        for pre in preproc_list:
            if (pre, sp) in table_map:
                metrics = table_map[(pre, sp)]
                group_rows.append({
                    'architecture': target_arch,
                    'preprocessing': pre,
                    'sparsity': sp,
                    'mIoU_mean': metrics['mIoU'][0], 'mIoU_std': metrics['mIoU'][1],
                    'mAcc_mean': metrics['mAcc'][0], 'mAcc_std': metrics['mAcc'][1],
                    'aAcc_mean': metrics['aAcc'][0], 'aAcc_std': metrics['aAcc'][1],
                })
        rows.extend(group_rows)
    return rows



# ---------- Erzeuge LaTeX-Tabelle ----------
def rows_to_latex_table(rows, caption, label):
    """
    rows: Liste von dicts in der Reihenfolge, wie sie in der Tabelle erscheinen sollen.
    Erwartet: erste row kann baseline (sparsity='-'), danach Gruppen (preprocessing repeated for each sparsity group).
    Generiert LaTeX-String.
    """
    if not rows:
        return "% Keine Daten für diese Konfiguration."
    arch = ARCH_NAME.get(rows[0]['architecture'], rows[0]['architecture'])
    # find baseline row (sparsity == '-'), assume first is baseline if present
    baseline = None
    baseline_group = []
    other_rows = []
    for r in rows:
        if r['sparsity'] == '-' and PRE_NAME.get(r['preprocessing'], r['preprocessing']) == '-':
            baseline = r
        elif r['sparsity'] == '-' and PRE_NAME.get(r['preprocessing'], r['preprocessing']) != '-':
            baseline_group.append(r)
        elif r['sparsity'] != '-' and PRE_NAME.get(r['preprocessing'], r['preprocessing']) != '-':
            other_rows.append(r)
    # group other_rows by sparsity in order
    grouped = OrderedDict()
    for r in other_rows:
        grouped.setdefault(r['sparsity'], []).append(r)
    # build latex
    header = r"""\begin{table}[h]
  \centering
  \caption{%s}
  \begin{tabular}{@{}lccc ccc@{}}
    \toprule
    Architecture & Preprocessing & Sparsity & mIoU & mAcc & aAcc \\
    \midrule""" % caption
    lines = [header]
    # baseline
    if baseline:
        line = "    %s & - & - & %s & %s & %s \\\\" % (
            arch,
            fmt_mean_std(baseline['mIoU_mean'], baseline['mIoU_std']),
            fmt_mean_std(baseline['mAcc_mean'], baseline['mAcc_std']),
            fmt_mean_std(baseline['aAcc_mean'], baseline['aAcc_std'])
        )
        lines.append(line)
    lines.append("    \\midrule")
    lines.append("    \\midrule")
    if baseline_group:
        n = len(baseline_group)
        first = baseline_group[0]
        line = "    \\multirow{%d}{*}{%s} & %s & \\multirow{%d}{*}{-} & %s & %s & %s \\\\" % (
            n,
            arch,
            PRE_NAME.get(first['preprocessing'], first['preprocessing']),
            n,
            fmt_mean_std(first['mIoU_mean'], first['mIoU_std']),
            fmt_mean_std(first['mAcc_mean'], first['mAcc_std']),
            fmt_mean_std(first['aAcc_mean'], first['aAcc_std'])
        )
        lines.append(line)
        for r in baseline_group[1:]:
            line = "    & %s & & %s & %s & %s \\\\" % (
                PRE_NAME.get(r['preprocessing'], r['preprocessing']),
                fmt_mean_std(r['mIoU_mean'], r['mIoU_std']),
                fmt_mean_std(r['mAcc_mean'], r['mAcc_std']),
                fmt_mean_std(r['aAcc_mean'], r['aAcc_std'])
            )
            lines.append(line)
        lines.append("    \\midrule")

    # grouped sparsity blocks
    def sparsity_sort_key(s):
        if s == '-':
            return -1.0
        # Entferne % und \, wandle in float
        try:
            return float(s.replace('\\','').replace('%',''))
        except:
            return 1000.0

    # Erzeuge neues OrderedDict nach aufsteigender Sparsity
    grouped_sorted = OrderedDict(sorted(grouped.items(), key=lambda x: sparsity_sort_key(x[0])))
    for sp, group in grouped_sorted.items():
        n = len(group)
        # first row: use multirow for architecture and sparsity
        first = group[0]
        line = "    \\multirow{%d}{*}{%s} & %s & \\multirow{%d}{*}{%s} & %s & %s & %s \\\\" % (
            n,
            arch,
            PRE_NAME.get(first['preprocessing'], first['preprocessing']),
            n,
            sp,
            fmt_mean_std(first['mIoU_mean'], first['mIoU_std']),
            fmt_mean_std(first['mAcc_mean'], first['mAcc_std']),
            fmt_mean_std(first['aAcc_mean'], first['aAcc_std'])
        )
        lines.append(line)
        # remaining rows in this group
        for r in group[1:]:
            line = "    & %s & & %s & %s & %s \\\\" % (
                PRE_NAME.get(r['preprocessing'], r['preprocessing']),
                fmt_mean_std(r['mIoU_mean'], r['mIoU_std']),
                fmt_mean_std(r['mAcc_mean'], r['mAcc_std']),
                fmt_mean_std(r['aAcc_mean'], r['aAcc_std'])
            )
            lines.append(line)
        lines.append("    \\midrule")
    footer = r"""    \bottomrule
  \end{tabular}
  \label{%s}
\end{table}""" % label
    lines.append(footer)
    return "\n".join(lines)

# ---------- Public helper: Erstellung einer Tabelle für gegebene Architektur und Test-Dataset ----------
def generate_latex_for(data_dir, architecture, test_dataset, caption=None, label=None, fill_missing=False):
    all_recs = load_all_results(data_dir)
    rows = build_table_records(all_recs, architecture, test_dataset, fill_missing_preproc_with_zeros=fill_missing)
    if caption is None:
        caption = f"Evaluation of the {architecture} architecture validated on {test_dataset} at different sparsities."
    if label is None:
        label = f"tab:sparse_results_{architecture}_{test_dataset}"
    latex = rows_to_latex_table(rows, caption, label)
    return latex

# ---------- Beispiel: wie man es benutzt ----------
if __name__ == "__main__":
    # Passe data_dir an dein Notebook / Dateiordner an:
    data_dir = "/home/lstracke/Visualization-Notebooks/trained_results"   # <-- hier deine txt-files ablegen

    # Architekturen
    architectures = ['mask2former', 'upernet', 'deeplabv3plus']
    # Test-Datasets
    test_datasets = ['acdc_full']

    for arch in architectures:
        for dataset in test_datasets:
            tex = generate_latex_for(
                data_dir,
                arch,
                dataset,
                caption=f"Evaluation of the {arch} architecture trained on Cityscapes and \\textbf{{validated on {dataset}}} at different sparsities.",
                label=f"tab:sparse_{arch}_{dataset}",
                fill_missing=(arch=='deeplabv3plus')  # fill zeros bei deeplabv3plus wenn gewünscht
            )
            out_path = f"./table_{arch}_{dataset}_threshold_acdc_full.tex"
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(tex)
            print(f"Wrote {out_path}")



Wrote ./table_mask2former_acdc_full_threshold_acdc_full.tex
Wrote ./table_upernet_acdc_full_threshold_acdc_full.tex
Wrote ./table_deeplabv3plus_acdc_full_threshold_acdc_full.tex
